# 02 — eModel Optimisation

Build an `EModelOptimizationScanConfig`, expand it via `GridScanGenerationTask`, and
run the `EModelOptimizationTask` for each coordinate. The task seeds ephys data +
`extracted_features.json` from the previous stage, copies the user-provided
morphology / mechanisms / params / recipes into the working directory, compiles
the mod files, merges the optimisation-related `pipeline_settings` from the blocks
into `recipes.json`, places the extracted features at the path the recipe expects
(`config/features/<emodel>.json` by default), then runs `pipeline.optimise(seed=...)`.

**Reads from:**
- `obi-output/01_efeature_extraction/grid_scan/0/` — `ephys_data/` and `extracted_features.json`.
- `obi-output/L_emodel_optimization_inputs/` — the L5PC recipes, morphology, mechanisms, and params.

**Writes to:** `obi-output/02_emodel_optimization/grid_scan/0/` — the working directory
plus the new `checkpoints/` directory and the `run/<iteration_tag>/` archive.

The example uses **`optimiser='SO-CMA'`** with very small `max_ngen=2` and
`offspring_size=4`. Increase those for production runs.


## Imports

In [1]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._02_emodel_optimization.blocks import (
    OptimizationInitialize,
    OptimizationParams,
    OptimizationSettings,
)


## Build the scan config

In [2]:
previous_stage = (
    Path.cwd() / "../../../obi-output/01_efeature_extraction/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 01_efeature_extraction.ipynb first."
)

inputs_root = (
    Path.cwd() / "../../../obi-output/L_emodel_optimization_inputs"
).resolve()
assert inputs_root.exists(), (
    f"{inputs_root} not found — run 00_setup_download_l5pc_data.ipynb first."
)

scan_config = obi.EModelOptimizationScanConfig(
    initialize=OptimizationInitialize(
        previous_stage_output_path=previous_stage,
        recipes_path=inputs_root / "config" / "recipes.json",
        morphology_path=inputs_root / "morphologies",
        mechanisms_path=inputs_root / "mechanisms",
        params_path=inputs_root / "config" / "params" / "pyr.json",
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    optimization_settings=OptimizationSettings(
        optimiser="SO-CMA",
        max_ngen=2,            # very small for demo runs
        optimisation_timeout=300.0,
        validation_threshold=5.0,
        seed=1,
    ),
    optimization_params=OptimizationParams(
        offspring_size=4,      # very small for demo runs
    ),
)
print(scan_config.optimization_settings.to_dict(scan_config.optimization_params))


{'optimiser': 'SO-CMA', 'max_ngen': 2, 'optimisation_timeout': 300.0, 'validation_threshold': 5.0, 'optimisation_params': {'offspring_size': 4}}


## Run the grid scan

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/02_emodel_optimization/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


[2026-06-05 07:37:08,810] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/ephys_data -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/ephys_data


[2026-06-05 07:37:08,833] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/extraction -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/extraction


[2026-06-05 07:37:08,834] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/extracted_features.json -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/extracted_features.json


[2026-06-05 07:37:08,841] INFO: Compiling NEURON mechanisms in /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/mechanisms with /Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/nrnivmodl ...


/usr/bin/xcrun
/Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0
Mod files: "mechanisms/mechanisms/Ca_HVA.mod" "mechanisms/mechanisms/Ca_HVA2.mod" "mechanisms/mechanisms/Ca_LVAst.mod" "mechanisms/mechanisms/CaDynamics_DC0.mod" "mechanisms/mechanisms/Ih.mod" "mechanisms/mechanisms/K_Pst.mod" "mechanisms/mechanisms/K_Tst.mod" "mechanisms/mechanisms/KdShu2007.mod" "mechanisms/mechanisms/Nap_Et2.mod" "mechanisms/mechanisms/NaTg.mod" "mechanisms/mechanisms/NaTg2.mod" "mechanisms/mechanisms/SK_E2.mod" "mechanisms/mechanisms/SKv3_1.mod" "mechanisms/mechanisms/StochKv2.mod" "mechanisms/mechanisms/StochKv3.mod"

Creating 'arm64' directory for .o files.

 -> Compiling mod_func.cpp
 -> NMODL ../mechanisms/Ca_HVA.mod
 -> NMODL ../mechanisms/Ca_LVAst.mod
 -> NMODL ../mechanisms/Ca_HVA2.mod
 -> NMODL ../mechanisms/CaDynamics_DC0.mod


...

[2026-06-05 07:45:45,767] INFO: 3  	4     	836.173	396.129	592.765	1521.92


[2026-06-05 07:45:45,769] INFO: CMA stopped because of termination criteria: M a x   n g e n


[2026-06-05 07:45:45,771] INFO: Running optimisation ... Done.


/Users/james/Documents/obi/code/obi-main/obi-one/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/james/Documents/obi/code/obi-main/obi-one/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


[PosixPath('/Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0')]

## Inspect the checkpoints

In [4]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

checkpoints = sorted((coord_root / "checkpoints").rglob("*.pkl"))
print(f"Checkpoint files ({len(checkpoints)}):")
for p in checkpoints:
    print(" ", p.relative_to(coord_root))


Coordinate output: /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0
Checkpoint files (1):
  checkpoints/L5PC/None/emodel=L5PC__species=rat__brain_region=SSCX__seed=1.pkl
